# 05 · Incremental — source→RAW, RAW→CLEAN (hoje × com DuckDB)

Testa o **incremental** nos estágios do pipeline e decide como fazer com o "tudo DuckDB".

| Estágio | Hoje | Com DuckDB (proposto) |
|---|---|---|
| **source → RAW** | cursor salvo à mão em `connector_state` + `write_delta(merge, primary_key)` (Delta) | `dlt.sources.incremental` + `write_disposition="merge"` (estado no `_dlt_pipeline_state`) |
| **RAW → CLEAN** | ❌ **não existe** — todos os 42 models são `materialized='table'` → **full-refresh sempre** | `materialized='incremental'` (merge por `unique_key` + watermark) |
| **CLEAN → grafo** | watermark `updated_at`; **se a coluna não existe → full scan** (bug, ver §colega) | mesma coisa — exige `updated_at` na clean |

**Foco:** medir **full × incremental** no RAW→CLEAN (1M), validar a preocupação do colega, e decidir.

> 📖 Decisão, colunas necessárias e "o que o usuário configura" no [RESULTADOS.md](RESULTADOS.md).

In [ ]:
# --- setup: simula o pipeline em DuckDB (source 1M com id + updated_at) ---
import sys; sys.path.insert(0, "/Users/allanfraga/Repos/strattum/experimentacoes")
import exp, duckdb, os, time

os.makedirs(f"{exp.DATA}/inc_test", exist_ok=True)
DB = f"{exp.DATA}/inc_test/inc.duckdb"
if os.path.exists(DB): os.remove(DB)
con = duckdb.connect(DB); con.execute("SET threads=8")

N = 1_000_000
con.execute(f"""CREATE TABLE source AS
  SELECT i AS id, 'user'||i||'@ACME.com' AS email, 'Customer '||i AS name,
         round((i%1000)*1.5, 2) AS balance, 'Sg'||(i%10) AS segment,
         TIMESTAMP '2026-01-01' + (i % 100)*INTERVAL 1 DAY AS updated_at
  FROM range({N}) t(i)""")

# O modelo CLEAN (mesmo SQL no full e no incremental; {where} muda por estágio).
# Carrega updated_at + external_id — as duas colunas que o incremental exige.
CLEAN = """SELECT id AS external_id, lower(trim(email)) AS email, trim(name) AS full_name,
  balance, segment,
  CASE WHEN balance>=1000 THEN 'high' WHEN balance>=100 THEN 'mid' ELSE 'low' END AS balance_tier,
  updated_at FROM {src} {where}"""

print("source:", f"{con.execute('SELECT count(*) FROM source').fetchone()[0]:,}", "linhas")

## 1ª carga (full) — popula RAW e CLEAN do zero

Primeira execução não tem watermark → carrega tudo. `source → raw` (merge por `id`) e
`raw → clean` (full). Guardamos o **watermark** (max `updated_at`) de cada camada pra próxima run.

In [ ]:
# source -> raw (1ª vez = tudo) ; raw -> clean (full)
con.execute("CREATE OR REPLACE TABLE raw AS SELECT * FROM source")
t = time.perf_counter()
con.execute(f"CREATE OR REPLACE TABLE clean AS {CLEAN.format(src='raw', where='')}")
t_full = (time.perf_counter() - t) * 1000

wm_raw = con.execute("SELECT max(updated_at) FROM raw").fetchone()[0]      # cursor source->raw
wm_clean = con.execute("SELECT max(updated_at) FROM clean").fetchone()[0]  # watermark raw->clean
print(f"raw={con.execute('SELECT count(*) FROM raw').fetchone()[0]:,} | "
      f"clean={con.execute('SELECT count(*) FROM clean').fetchone()[0]:,} | "
      f"FULL raw→clean: {t_full:.0f}ms")
print("watermarks:", wm_raw, "(raw) /", wm_clean, "(clean)")

## 2ª carga (incremental) — só o delta

Chega um delta na fonte (**10k atualizados + 10k novos**). O incremental processa só isso:
1. **source → raw:** `WHERE updated_at > wm_raw` (cursor) + `MERGE` por `id`;
2. **raw → clean:** `WHERE updated_at > wm_clean` (watermark) + `MERGE` por `external_id`
   (**UPDATE** existentes, **INSERT** novos).

Comparamos com o **full rebuild** e medimos o **skip** (se nada mudou).

In [ ]:
# delta na fonte: 10k atualizados + 10k novos (todos mais novos que o watermark)
con.execute("UPDATE source SET balance=balance+1, updated_at=TIMESTAMP '2026-06-01' WHERE id < 10000")
con.execute("INSERT INTO source SELECT 5000000+i,'new'||i||'@x.com','New '||i,5.0,'Sg1',"
            "TIMESTAMP '2026-06-01' FROM range(10000) t(i)")

# 1) source -> raw : cursor (updated_at > wm) + merge por id
con.execute(f"""MERGE INTO raw USING (SELECT * FROM source WHERE updated_at > '{wm_raw}') s
  ON raw.id = s.id WHEN MATCHED THEN UPDATE SET email=s.email, name=s.name, balance=s.balance,
     segment=s.segment, updated_at=s.updated_at
  WHEN NOT MATCHED THEN INSERT VALUES (s.id, s.email, s.name, s.balance, s.segment, s.updated_at)""")

# 2) raw -> clean INCREMENTAL : watermark + merge por external_id (UPDATE existentes / INSERT novos)
t = time.perf_counter()
con.execute(f"""MERGE INTO clean USING ({CLEAN.format(src='raw', where=f"WHERE updated_at > '{wm_clean}'")}) src
  ON clean.external_id = src.external_id
  WHEN MATCHED THEN UPDATE SET email=src.email, full_name=src.full_name, balance=src.balance,
     segment=src.segment, balance_tier=src.balance_tier, updated_at=src.updated_at
  WHEN NOT MATCHED THEN INSERT VALUES (src.external_id, src.email, src.full_name, src.balance,
     src.segment, src.balance_tier, src.updated_at)""")
t_inc = (time.perf_counter() - t) * 1000
delta_n = con.execute(f"SELECT count(*) FROM raw WHERE updated_at > '{wm_clean}'").fetchone()[0]

# comparação: full rebuild (reprocessa tudo)
t = time.perf_counter()
con.execute(f"CREATE OR REPLACE TABLE clean_full AS {CLEAN.format(src='raw', where='')}")
t_full2 = (time.perf_counter() - t) * 1000

# skip: nada mudou -> delta = 0 -> pula o dbt
wm2 = con.execute("SELECT max(updated_at) FROM clean").fetchone()[0]
t = time.perf_counter()
changed = con.execute(f"SELECT count(*) FROM raw WHERE updated_at > '{wm2}'").fetchone()[0]
t_skip = (time.perf_counter() - t) * 1000

print(f"INCREMENTAL (merge {delta_n:,}): {t_inc:.0f}ms | clean = {con.execute('SELECT count(*) FROM clean').fetchone()[0]:,}")
print(f"FULL rebuild (tudo):           {t_full2:.0f}ms")
print(f"speedup incremental:           {t_full2/t_inc:.0f}x")
print(f"SKIP check (nada mudou):       delta={changed}, {t_skip:.1f}ms -> pula o dbt")

## A preocupação do colega — **válida e confirmada no código**

> *"Tabelas clean sem `updated_at` são re-lidas inteiras a cada rodada (full scan). Causa raiz:
> `CleanReader` só aplica `WHERE updated_at > watermark` se a coluna existir."*

✅ **Procede.** Confirmado em [`reader.py:232-246`](../../strattum-ai/services/memory-worker/memory_worker/reader.py#L232):
se `updated_at` não está nas colunas, ele **pula o filtro e faz `SELECT *`** (full scan), e o
memory_worker reprocessa a tabela inteira → re-duplica (nós com `entity_id` não-determinístico,
ou eventos com `CREATE` em vez de `MERGE`).

É um **gatilho real** porque **24 dos 42 clean models não têm `updated_at`** (ex.: `asaas_customers`,
`zendesk_ticket_comments` + edge tables). A célula abaixo audita isso.

**Como o "tudo DuckDB" conserta?** Não conserta sozinho — o bug é de **coluna**, não de formato. A
correção é a **mesma** que habilita o incremental do RAW→CLEAN: **garantir `updated_at` na clean**.
Resolve os dois de uma vez.

In [ ]:
# auditoria: quais dos 42 clean models reais carregam updated_at?
import glob, re
sqls = glob.glob(f"{exp.REPO}/strattum-data/services/pipelines/src/connectors/*/transforms/*.sql")
com, sem = [], []
for f in sqls:
    txt = open(f).read().lower()
    (com if re.search(r"as updated_at|\bupdated_at\b", txt) else sem).append(os.path.basename(f))
print(f"clean models: {len(sqls)} | COM updated_at: {len(com)} | SEM: {len(sem)}")
print("SEM updated_at (fazem full scan no grafo):", ", ".join(sorted(sem)[:8]), "...")

# demo do comportamento do reader.py: sem a coluna, o WHERE é descartado
k = duckdb.connect()
k.execute("CREATE TABLE sem_ts AS SELECT 1 id UNION ALL SELECT 2")
cols = [r[1] for r in k.execute("PRAGMA table_info('sem_ts')").fetchall()]
sql = ("SELECT * FROM sem_ts WHERE updated_at > '2026-01-01'" if "updated_at" in cols
       else "SELECT * FROM sem_ts   -- updated_at ausente -> FULL SCAN (watermark ignorado)")
print("\nquery efetiva do reader:", sql, "->", k.sql(sql).fetchall())

## Resultados (1M) e colunas necessárias

| | tempo | observação |
|---|---|---|
| **RAW→CLEAN full** | ~160 ms | reprocessa 1M sempre |
| **RAW→CLEAN incremental** (delta 20k) | ~20 ms | **~8× a 2% de delta**; o ganho cresce com a tabela (delta fixo) |
| **skip check** (nada mudou) | ~0,2 ms | conta o delta; se 0 → **pula o dbt** |

**Duas colunas habilitam tudo** (incremental + watermark do grafo):

| Coluna | Pra quê | Exemplo real |
|---|---|---|
| **`external_id`** (PK / `unique_key`) | chave do `MERGE` | ✅ todos os 42 (`c.id AS external_id`) |
| **`updated_at`** (watermark) | filtro do delta + `unique_key` do dbt | ✅ 18/42 (`hubspot`: `lastmodifieddate AS updated_at`); ❌ 24/42 (`asaas`, edges) |

> 📖 **Decisão, o que o usuário configura, e o plano (merge + skip) no [RESULTADOS.md](RESULTADOS.md).**